# Marketing Copy Generator with Eden AI

Drop a product brief → 4 LLMs each return: 3 taglines, 2 LinkedIn posts, 5 SEO title variants. Pick a tone (punchy / professional / playful), see which model gets it.

Same arena pattern, applied to creative generation. The interesting compare here isn't *correctness* (everyone produces text) but **style fidelity, length compliance, and originality** — exactly the dimensions you care about for copy.

**Prerequisites:** an Eden AI API key (set as `EDENAI_API_KEY` env var or in `.env`).

In [ ]:
%pip install --quiet aiohttp ipywidgets nest_asyncio python-dotenv

## 1. Configuration

In [ ]:
import base64
import json
import os

from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(override=True)

EDENAI_API_KEY = os.environ.get("EDENAI_API_KEY")
if not EDENAI_API_KEY:
    raise RuntimeError("Set EDENAI_API_KEY (env var or .env file). Get one at https://app.edenai.run")
EDENAI_URL = "https://api.edenai.run/v3/llm/chat/completions"

MODELS = [
    {"label": "Claude",  "model": "anthropic/claude-sonnet-4-5"},
    {"label": "GPT-4o",  "model": "openai/gpt-4o"},
    {"label": "Gemini",  "model": "google/gemini-2.5-flash"},
    {"label": "Mistral", "model": "mistral/mistral-large-latest"},
]

SAMPLE_PRODUCTS = {
    "SaaS — observability": {
        "name": "Cinder",
        "description": "An observability platform for ML pipelines. Surfaces data drift, model regressions, and prompt-injection attempts in real time.",
        "audience": "ML engineering leads at series-B startups",
    },
    "DTC — consumer": {
        "name": "Bloom Bottle",
        "description": "A self-cleaning water bottle. UV-C light sterilizes the inside every 4 hours. 24h cold retention, dishwasher safe lid.",
        "audience": "health-conscious urban professionals 25–40",
    },
    "Mobile app": {
        "name": "Habitica Pro",
        "description": "A habit tracker that gamifies your goals with role-playing mechanics — earn XP, level up, get gear.",
        "audience": "gamers who want to be more productive",
    },
}

TONES = ["punchy", "professional", "playful", "technical"]


def _is_sandbox(jwt: str) -> bool:
    try:
        payload_b64 = jwt.split(".")[1]
        payload_b64 += "=" * (-len(payload_b64) % 4)
        return json.loads(base64.urlsafe_b64decode(payload_b64)).get("type") == "sandbox_api_token"
    except Exception:
        return False


if _is_sandbox(EDENAI_API_KEY):
    display(HTML(
        '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 14px;'
        'border-radius:4px;font-family:sans-serif;font-size:13px;margin:6px 0;">'
        '<b>⚠ Sandbox key detected.</b> Use a production key for real model comparison.</div>'
    ))

## 2. The generator

One call per model, asking for all 10 pieces of copy in a single structured JSON response. The prompt encodes the constraints (tagline ≤ 8 words, LinkedIn ≤ 60 words, SEO title ≤ 60 chars) — we then post-validate to see which models obey.

In [ ]:
import asyncio
import time

import aiohttp

MAX_RETRIES = 2

TAGLINE_MAX_WORDS = 8
LINKEDIN_MAX_WORDS = 60
SEO_MAX_CHARS = 60


def _build_prompt(product, tone):
    return (
        f"Generate marketing copy for the following product, in a {tone} tone aimed at "
        f"the target audience. Return ONLY a JSON object (no prose, no markdown fences) with keys:\n"
        "  - taglines: array of EXACTLY 3 strings, each ≤ 8 words\n"
        "  - linkedin_posts: array of EXACTLY 2 strings, each ≤ 60 words\n"
        "  - seo_titles: array of EXACTLY 5 strings, each ≤ 60 characters\n\n"
        f"PRODUCT: {product['name']}\n"
        f"DESCRIPTION: {product['description']}\n"
        f"AUDIENCE: {product['audience']}\n"
        f"TONE: {tone}"
    )


def _strip_fences(text):
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1] if "\n" in t else t[3:]
        if t.endswith("```"):
            t = t.rsplit("```", 1)[0]
    return t.strip()


async def _call_llm(session, payload):
    headers = {"Authorization": f"Bearer {EDENAI_API_KEY}", "Content-Type": "application/json"}
    for attempt in range(MAX_RETRIES + 1):
        async with session.post(EDENAI_URL, headers=headers, json=payload,
                                timeout=aiohttp.ClientTimeout(total=90)) as resp:
            body = await resp.text()
            if resp.status == 200:
                return json.loads(body)
            if resp.status in (400, 429, 502, 503, 504) and attempt < MAX_RETRIES:
                await asyncio.sleep(0.6 * (attempt + 1))
                continue
            raise RuntimeError(f"HTTP {resp.status}: {body[:200]}")
    raise RuntimeError("exhausted retries")


def _validate(parsed):
    issues = []
    tags = parsed.get("taglines") or []
    if len(tags) != 3:
        issues.append(f"got {len(tags)} taglines, expected 3")
    for i, t in enumerate(tags):
        words = len(str(t).split())
        if words > TAGLINE_MAX_WORDS:
            issues.append(f"tagline #{i+1} has {words} words (max {TAGLINE_MAX_WORDS})")
    posts = parsed.get("linkedin_posts") or []
    if len(posts) != 2:
        issues.append(f"got {len(posts)} LinkedIn posts, expected 2")
    for i, p in enumerate(posts):
        words = len(str(p).split())
        if words > LINKEDIN_MAX_WORDS:
            issues.append(f"LinkedIn #{i+1} has {words} words (max {LINKEDIN_MAX_WORDS})")
    titles = parsed.get("seo_titles") or []
    if len(titles) != 5:
        issues.append(f"got {len(titles)} SEO titles, expected 5")
    for i, t in enumerate(titles):
        if len(str(t)) > SEO_MAX_CHARS:
            issues.append(f"SEO title #{i+1} is {len(str(t))} chars (max {SEO_MAX_CHARS})")
    return issues


async def generate_one(session, model_cfg, product, tone):
    payload = {
        "model": model_cfg["model"],
        "messages": [{"role": "user", "content": _build_prompt(product, tone)}],
    }
    start = time.perf_counter()
    try:
        data = await _call_llm(session, payload)
        content = data["choices"][0]["message"]["content"]
    except Exception as e:
        return {"label": model_cfg["label"], "status": "error",
                "latency": time.perf_counter() - start, "error": str(e)}
    raw = _strip_fences(content)
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as e:
        return {"label": model_cfg["label"], "status": "invalid_json",
                "latency": time.perf_counter() - start, "raw": raw[:600], "error": str(e)}
    issues = _validate(parsed)
    return {
        "label": model_cfg["label"], "status": "ok" if not issues else "length_violation",
        "latency": time.perf_counter() - start, "parsed": parsed, "issues": issues,
    }

## 3. UI

In [ ]:
from ipywidgets import (
    Button, Dropdown, GridBox, HBox, HTML as HTMLWidget,
    Layout, Output, Text, Textarea, VBox,
)
from IPython.display import display

sample_dropdown = Dropdown(
    options=list(SAMPLE_PRODUCTS.keys()),
    value="SaaS — observability",
    description="Sample:",
    layout=Layout(width="320px"),
)
name_box = Text(value=SAMPLE_PRODUCTS[sample_dropdown.value]["name"], description="Name:", layout=Layout(width="600px"))
audience_box = Text(value=SAMPLE_PRODUCTS[sample_dropdown.value]["audience"], description="Audience:", layout=Layout(width="600px"))
desc_box = Textarea(value=SAMPLE_PRODUCTS[sample_dropdown.value]["description"], description="Description:", layout=Layout(width="600px", height="60px"))
tone_dropdown = Dropdown(options=TONES, value="punchy", description="Tone:", layout=Layout(width="240px"))


def _on_sample_change(change):
    if change["name"] == "value":
        p = SAMPLE_PRODUCTS[change["new"]]
        name_box.value = p["name"]
        audience_box.value = p["audience"]
        desc_box.value = p["description"]


sample_dropdown.observe(_on_sample_change, names="value")

generate_btn = Button(description="✍ Generate", button_style="primary")
clear_btn = Button(description="Clear")

panel_layout = Layout(border="1px solid #ddd", padding="8px", height="420px", overflow="auto")
panels = [Output(layout=panel_layout) for _ in MODELS]
headers = [HTMLWidget() for _ in MODELS]


def _empty_header(i):
    m = MODELS[i]
    return (
        f'<div style="font-family:sans-serif;font-size:13px;padding:4px;">'
        f'<b>{m["label"]}</b> <span style="color:#888;font-size:11px;">{m["model"]}</span></div>'
    )


def _set_empty_panel(i):
    headers[i].value = _empty_header(i)
    panels[i].clear_output()
    with panels[i]:
        display(HTML(
            '<div style="font-family:sans-serif;color:#aaa;font-size:12px;text-align:center;padding:70px 10px;">'
            'Pick a product, tone, and click<br><b>✍ Generate</b></div>'
        ))


for i in range(len(MODELS)):
    _set_empty_panel(i)

panel_blocks = [
    VBox([headers[i], panels[i]], layout=Layout(border="1px solid #eee", padding="4px", border_radius="4px"))
    for i in range(len(MODELS))
]
grid = GridBox(
    panel_blocks,
    layout=Layout(grid_template_columns="repeat(2, 1fr)", grid_gap="8px"),
)

summary_out = Output()

display(VBox([
    HBox([sample_dropdown, tone_dropdown]),
    name_box, audience_box, desc_box,
    HBox([generate_btn, clear_btn]),
    grid,
    summary_out,
]))

## 4. Wire it up

In [ ]:
import html as _html

import nest_asyncio
from IPython.display import clear_output

nest_asyncio.apply()

display(HTML('''
<style>
@keyframes cb_blink { 0%, 100% { opacity: 0.2; } 50% { opacity: 1; } }
.cb-dot { animation: cb_blink 1.2s infinite both; display:inline-block; }
.cb-dot:nth-child(2) { animation-delay: 0.2s; }
.cb-dot:nth-child(3) { animation-delay: 0.4s; }
</style>
'''))

STATUS_STYLES = {
    "ok":                ("all constraints ✓", "#28a745"),
    "length_violation":  ("length issue ⚠",    "#fd7e14"),
    "invalid_json":      ("invalid JSON ✗",    "#dc3545"),
    "error":             ("error ✗",           "#dc3545"),
}


def _render_header(i, status, latency, issue_count=None):
    m = MODELS[i]
    status_label, color = STATUS_STYLES.get(status, (status, "#6c757d"))
    issues_label = f' · {issue_count} issue{"s" if issue_count != 1 else ""}' if issue_count else ''
    headers[i].value = (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px;">'
        f'  <div><b style="font-size:13px;">{m["label"]}</b> '
        f'<span style="color:#888;font-size:11px;">{m["model"]}</span></div>'
        f'  <div><span style="background:{color};color:white;padding:3px 10px;'
        f'border-radius:10px;font-size:11px;font-weight:600;">'
        f'{status_label} · {latency:.2f}s{issues_label}</span></div>'
        '</div>'
    )


def _set_loading_header(i):
    m = MODELS[i]
    headers[i].value = (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px;">'
        f'  <div><b style="font-size:13px;">{m["label"]}</b> '
        f'<span style="color:#888;font-size:11px;">{m["model"]}</span></div>'
        '  <div><span style="background:#17a2b8;color:white;padding:3px 10px;'
        'border-radius:10px;font-size:11px;font-weight:600;">writing'
        '<span class="cb-dot">.</span><span class="cb-dot">.</span><span class="cb-dot">.</span></span></div>'
        '</div>'
    )


def _copy_button(text, label="Copy"):
    safe = text.replace("\\", "\\\\").replace("`", "\\`").replace("</", "<\\/")
    return (
        f'<button onclick="navigator.clipboard.writeText(`{safe}`);'
        f'this.textContent=&quot;✓&quot;;setTimeout(()=>this.textContent=&quot;{label}&quot;,1200);" '
        f'style="font-size:10px;padding:1px 6px;border:1px solid #ddd;background:#fff;'
        f'border-radius:3px;cursor:pointer;margin-left:6px;">{label}</button>'
    )


def _section(title, items, max_words, char_limit=None):
    rows = ""
    for item in items:
        text = str(item)
        words = len(text.split())
        if char_limit:
            over = len(text) > char_limit
            counter = f'{len(text)} chars'
        else:
            over = words > max_words
            counter = f'{words} words'
        counter_color = "#dc3545" if over else "#888"
        rows += (
            f'<div style="display:flex;justify-content:space-between;align-items:start;'
            f'padding:4px 0;border-bottom:1px solid #f0f0f0;">'
            f'<div style="font-family:sans-serif;font-size:12px;flex:1;">{_html.escape(text)}</div>'
            f'<div style="font-family:monospace;font-size:10px;color:{counter_color};white-space:nowrap;">{counter}{_copy_button(text)}</div>'
            f'</div>'
        )
    return (
        f'<div style="font-family:sans-serif;font-size:11px;color:#666;'
        f'margin-top:8px;text-transform:uppercase;letter-spacing:0.5px;"><b>{title}</b></div>'
        f'{rows}'
    )


def _render_panel(i, result):
    status = result["status"]
    issue_count = len(result.get("issues") or [])
    _render_header(i, status, result["latency"], issue_count)
    panels[i].clear_output()
    with panels[i]:
        if status in ("ok", "length_violation"):
            p = result["parsed"]
            issues_html = ""
            if result.get("issues"):
                items = "".join(f'<li style="color:#fd7e14;">{i}</li>' for i in result["issues"])
                issues_html = (
                    f'<div style="background:#fff3cd;padding:6px 10px;border-radius:3px;'
                    f'font-family:sans-serif;font-size:11px;margin-bottom:6px;">'
                    f'<b>Constraint issues:</b><ul style="margin:4px 0;padding-left:18px;">{items}</ul></div>'
                )
            display(HTML(
                issues_html +
                _section("Taglines (≤ 8 words)", p.get("taglines") or [], 8) +
                _section("LinkedIn posts (≤ 60 words)", p.get("linkedin_posts") or [], 60) +
                _section("SEO titles (≤ 60 chars)", p.get("seo_titles") or [], 0, char_limit=60)
            ))
        else:
            err = _html.escape(str(result.get("error", "unknown")))[:400]
            display(HTML(
                f'<div style="color:#dc3545;font-family:monospace;font-size:11px;padding:8px;'
                f'background:#f8d7da;border-radius:4px;">{err}</div>'
            ))


def _render_summary(results):
    fastest = min(results, key=lambda r: r["latency"])
    compliant = [r["label"] for r in results if r["status"] == "ok"]
    violators = [(r["label"], len(r.get("issues") or [])) for r in results if r["status"] == "length_violation"]
    failed = [r["label"] for r in results if r["status"] not in ("ok", "length_violation")]
    parts = [
        f'<span style="color:#666;">⚡ Fastest: <b>{fastest["label"]}</b> ({fastest["latency"]:.2f}s)</span>',
    ]
    if compliant:
        parts.append(f'<span style="color:#28a745;">✓ Followed length rules: {", ".join(compliant)}</span>')
    if violators:
        parts.append(f'<span style="color:#fd7e14;">⚠ Broke constraints: '
                     + ", ".join(f"{l} ({n})" for l, n in violators) + '</span>')
    if failed:
        parts.append(f'<span style="color:#dc3545;">✗ Failed: {", ".join(failed)}</span>')
    with summary_out:
        clear_output()
        display(HTML(
            f'<div style="background:#f8f9fa;padding:10px 12px;border-radius:4px;'
            f'border-left:4px solid #007bff;font-family:sans-serif;font-size:13px;'
            f'display:flex;gap:18px;flex-wrap:wrap;">{"".join(parts)}</div>'
        ))


async def run_round(product, tone):
    for i in range(len(MODELS)):
        panels[i].clear_output()
        with panels[i]:
            display(HTML(
                '<div style="font-family:sans-serif;color:#17a2b8;font-size:13px;text-align:center;padding:70px 10px;">'
                'writing<span class="cb-dot">.</span><span class="cb-dot">.</span><span class="cb-dot">.</span></div>'
            ))
        _set_loading_header(i)
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(*[
            generate_one(session, MODELS[i], product, tone) for i in range(len(MODELS))
        ])
    for i, r in enumerate(results):
        _render_panel(i, r)
    _render_summary(results)
    return results


def on_generate(_):
    product = {
        "name": name_box.value.strip(),
        "description": desc_box.value.strip(),
        "audience": audience_box.value.strip(),
    }
    if not product["name"] or not product["description"]:
        return
    asyncio.run(run_round(product, tone_dropdown.value))


def on_clear(_):
    for i in range(len(MODELS)):
        _set_empty_panel(i)
    summary_out.clear_output()


generate_btn.on_click(on_generate)
clear_btn.on_click(on_clear)

## 5. The failure modes worth watching

- **Length compliance** — the prompt explicitly says "≤ 8 words" for taglines. Most models obey, some don't. The word/char counter next to each output flags the overshoot in red.
- **Tone fidelity** — change the dropdown from "punchy" to "professional" and see how much each model actually shifts. Some keep the same energy regardless.
- **Repetition** — does the model produce 3 *meaningfully different* taglines, or 3 paraphrases of the same idea? Eyeball test.
- **Format compliance** — sometimes models return a paragraph instead of a list, or wrap JSON in markdown fences. The error panel surfaces parsing failures.

## 6. Customize

**Different copy formats.** Edit `_build_prompt` to ask for email subject lines, Twitter threads, ad headlines, product hero copy, etc.

**A/B sourcing.** Loop the same product through 4 models, keep only the highest-rated variants (judged by a 5th LLM):

```python
all_variants = []
for model in MODELS:
    result = await generate_one(session, model, product, tone)
    all_variants.extend(result['parsed']['taglines'])
# Then pipe all_variants through a judge prompt: "rank these 12 taglines by [criteria]"
```

Get 4× more variants than a single model, then let a judge LLM pick the top 3.